# Return distribution

Moments, quantiles, tail behaviour against a Gaussian reference, and normality tests read with the caution large samples demand.

**This notebook contains no calculations.** Everything is computed by
`xauusd_quant.research` and read back from
`results/statistical_research/<timeframe>/`. Generate those first:

```
xq research-summary --timeframe 5m
xq research-summary --all
```

Nothing here is a trading signal. No transaction costs are applied anywhere.

In [ ]:
import sys
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl

from xauusd_quant import load_config
from xauusd_quant.research import load_research_config

config = load_config(ROOT / "config" / "data.yaml")
research = load_research_config(ROOT / "config" / "research.yaml")

TIMEFRAME = "5m"          # change and re-run

print(f"instrument : {config.instrument}")
print(f"timezone   : {config.timezone.describe()}")
print(f"time basis : {research.intraday.time_basis}")
print(f"results    : {research.results_path}")

In [ ]:
import json

results = research.results_dir(TIMEFRAME)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME}. Run:  xq research-summary --timeframe {TIMEFRAME}"
    )

def table(name):
    """Load a previously-generated research table."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
print(f"generated : {summary['provenance']['generated_utc']}")
print(f"span      : {summary['bars']['first_timestamp']} -> {summary['bars']['last_timestamp']}")
print(f"bars      : {summary['bars']['count']:,}")

## Descriptive statistics

In [ ]:
dist = summary["distribution"]
for key in ("count", "mean", "median", "std", "skewness", "excess_kurtosis"):
    print(f"{key:18}: {dist[key]}")
print()
print("Excess kurtosis is relative to the Gaussian: 0 = Gaussian tails.")

## Quantiles

In [ ]:
pl.DataFrame([dist['quantiles']]).transpose(
    include_header=True, header_name='quantile', column_names=['log_return']
)

## How heavy are the tails?

`ratio_observed_to_gaussian` is the honest measure: how many times more often a move of that size actually happened than a Gaussian would allow.

In [ ]:
tails = table('gaussian_tail_comparison')
tails

## Normality tests

With this many observations these tests reject essentially always. Read the effect sizes above, not the p-values.

In [ ]:
norm = dist["normality"]
for test in norm["tests"]:
    print(f"{test['test']:15} statistic={test['statistic']:.4g}  p={test['p_value']:.3g}  "
          f"rejects={test['reject_at_5pct']}")
print()
if norm["large_sample_warning"]:
    print("WARNING:", norm["large_sample_warning"])
print()
print(norm["interpretation"])

## Plots

Generated by `xauusd_quant.research.plots`.

In [ ]:
from IPython.display import Image, display

for name in ("return_histogram", "qq_plot"):
    path = research.plots_dir(TIMEFRAME) / f"{name}.png"
    if path.exists():
        display(Image(str(path)))